# Import des données d'achat sur le S&P 500 et le Gold (via Excel), ainsi que STEF et CiC (via relevé de compte)

In [89]:
import pandas as pd
from sqlalchemy import create_engine
import config # fichier config.py
from pandas.tseries.offsets import MonthEnd

In [90]:
nom_fichier = r"E:\OneDrive\Documents\Programmation\Excel\Investissements.xlsb.xlsx"
df_sp500 = pd.read_excel(nom_fichier, sheet_name='S&P 500')
df_gold = pd.read_excel(nom_fichier, sheet_name='Gold')
df_livret = pd.read_excel(nom_fichier, sheet_name='Epargne')

In [91]:
# Création de l'engine
url = f"postgresql://{config.db_params['user']}:{config.db_params['password']}@{config.db_params['host']}:{config.db_params['port']}/{config.db_params['database']}"
engine = create_engine(url)

## Preparation et import des fichiers historiques

### S&P 500

In [92]:
df_sp500 = df_sp500[['S&P 500', 'Prix', 'Parts', 'Frais']]

df_sp500 = df_sp500.rename(columns={
    'S&P 500' : 'mvt_date', 
    'Prix' : 'mvt_prix', 
    'Parts' : 'mvt_nb_parts', 
    'Frais' : 'mvt_frais'})

df_sp500['mvt_type_mouvement'] = 'ACHAT'
df_sp500['pdt_id'] = 1

df_sp500.head()

,mvt_date,mvt_prix,mvt_nb_parts,mvt_frais,mvt_type_mouvement,pdt_id
0,2024-01-30,22.5829,17,1.919546,ACHAT,1
1,2024-02-16,23.2870,17,1.979395,ACHAT,1
2,2024-03-05,23.2302,17,1.974567,ACHAT,1
3,2024-04-08,23.9054,17,1.990000,ACHAT,1
4,2024-05-14,24.1027,16,1.928216,ACHAT,1


In [93]:
df_sp500.to_sql('mouvement_mvt', engine, if_exists='append', index=False)
print("S&P 500 importé !")

S&P 500 importé !


### Gold

In [94]:
df_gold = df_gold[['Gold', 'Prix', 'Parts', 'Frais']]

df_gold = df_gold.rename(columns={
    'Gold' : 'mvt_date', 
    'Prix' : 'mvt_prix', 
    'Parts' : 'mvt_nb_parts', 
    'Frais' : 'mvt_frais'})

df_gold['mvt_frais'] = df_gold['mvt_frais'].fillna(0)

df_gold['mvt_type_mouvement'] = 'ACHAT'
df_gold['pdt_id'] = 2

df_gold.head()

,mvt_date,mvt_prix,mvt_nb_parts,mvt_frais,mvt_type_mouvement,pdt_id
0,2024-01-30,36.486,2.685961,0.99,ACHAT,2
1,2024-02-16,36.258,2.758012,0.00,ACHAT,2
2,2024-03-18,38.488,2.598212,0.00,ACHAT,2
3,2024-04-16,43.480,2.299908,0.00,ACHAT,2
4,2024-05-16,42.718,2.340933,0.00,ACHAT,2


In [95]:
df_gold.to_sql('mouvement_mvt', engine, if_exists='append', index=False)
print("Gold importé !")

Gold importé !


### STEF

In [96]:
cols = ['mvt_date', 'mvt_prix', 'mvt_nb_parts', 'mvt_type_mouvement']

data_stef = [
    ['2023-05-05', 72.3280, 10.7427, 'ACHAT'],
    ['2023-05-05', 72.3280, 2.4252, 'ABONDEMENT'],
    ['2023-05-03', 72.3280, 18.2813, 'ACHAT'],
    ['2023-05-03', 72.3280, 7.8725, 'ABONDEMENT'],
    ['2022-05-06', 60.5550, 18.2413, 'ACHAT'],
    ['2022-05-06', 60.5550, 4.1181, 'ABONDEMENT'],
    ['2022-05-05', 60.5550, 22.2206, 'ACHAT'],
    ['2022-05-05', 60.5550, 9.4901, 'ABONDEMENT'],
    ['2021-04-30', 59.6840, 21.5597, 'ACHAT'],
    ['2021-04-30', 59.6840, 9.4060, 'ABONDEMENT']
    ]

df_stef = pd.DataFrame(data_stef, columns=cols)

df_stef['pdt_id'] = 3
df_stef['mvt_date'] = pd.to_datetime(df_stef['mvt_date'])

In [97]:
df_stef.to_sql('mouvement_mvt', engine, if_exists='append', index=False)
print("STEF importé !")

STEF importé !


### CiC

In [98]:
cols = ['mvt_date', 'mvt_prix', 'mvt_nb_parts', 'mvt_type_mouvement', 'pdt_id']

data_cic = [
    ['2019-03-14', 4.1108, 121.6308, 'ACHAT', 9],
    ['2019-03-14', 4.1108, 329.4979, 'ABONDEMENT', 9],
    ['2020-03-24', 3.6784, 271.8573, 'ACHAT', 4],
    ['2020-03-24', 3.6784, 736.4615, 'ABONDEMENT', 4],
    ['2022-02-07', 4.0336, -87.3620, 'VENTE', 9],
    ['2022-02-22', 13.5496, 29.5211, 'ACHAT', 5],
    ['2022-02-22', 20.1317, 29.8037, 'ACHAT', 6],
    ['2022-02-22', 13.5496, 79.9728, 'ABONDEMENT', 5],
    ['2022-02-22', 20.1317, 80.7383, 'ABONDEMENT', 6],
    ['2022-02-24', 4.0317, -451.1287, 'VENTE', 9],
    ['2022-02-24', 13.3427, 136.3157, 'ACHAT', 5],
    ]

df_cic = pd.DataFrame(data_cic, columns=cols)

df_cic['mvt_date'] = pd.to_datetime(df_cic['mvt_date'])

In [99]:
df_cic.to_sql('mouvement_mvt', engine, if_exists='append', index=False)
print("CiC importé !")

CiC importé !


### Livret A

In [100]:
df_la = df_livret[['Fin Mois', 'Livret A']]

df_la = df_la.rename(columns={
    'Fin Mois' : 'mvt_date', 
    'Livret A' : 'solde'})

df_la['mvt_prix'] = 1
df_la['mvt_nb_parts'] = df_la['solde'].diff().fillna(df_la['solde'])
df_la['mvt_frais'] = 0
# Typage automatique selon le signe
df_la.loc[df_la['mvt_nb_parts'] > 0, 'mvt_type_mouvement'] = 'APPORT'
df_la.loc[df_la['mvt_nb_parts'] < 0, 'mvt_type_mouvement'] = 'RETRAIT'

df_la['pdt_id'] = 10
del df_la['solde']
# On ne garde que ce qui a bougé
df_la = df_la[df_la['mvt_nb_parts'] != 0]

# On force la colonne en datetime (si ce n'est pas déjà fait)
df_la['mvt_date'] = pd.to_datetime(df_la['mvt_date'])

# On décale au dernier jour du mois
df_la['mvt_date'] = df_la['mvt_date'] + MonthEnd(0)

In [101]:
df_la.to_sql('mouvement_mvt', engine, if_exists='append', index=False)
print("Livret A importé !")

Livret A importé !


### PEL

In [102]:
df_lep = df_livret[['Fin Mois', 'LEP']]

df_lep = df_lep.rename(columns={
    'Fin Mois' : 'mvt_date', 
    'LEP' : 'solde'})

df_lep['mvt_prix'] = 1
df_lep['mvt_nb_parts'] = df_lep['solde'].diff().fillna(df_lep['solde'])
df_lep['mvt_frais'] = 0
# Typage automatique selon le signe
df_lep.loc[df_lep['mvt_nb_parts'] > 0, 'mvt_type_mouvement'] = 'APPORT'
df_lep.loc[df_lep['mvt_nb_parts'] < 0, 'mvt_type_mouvement'] = 'RETRAIT'

df_lep['pdt_id'] = 11
del df_lep['solde']
# On ne garde que ce qui a bougé
df_lep = df_lep[df_lep['mvt_nb_parts'] != 0]

# 1. On cible la ligne de décembre qui contient le "trop-plein"
date_decembre = '2024-12-01'
idx_dec = df_lep[df_lep['mvt_date'] == date_decembre].index

if not idx_dec.empty:
    # 2. On fige l'apport de décembre à 500€
    df_lep.loc[idx_dec, 'mvt_nb_parts'] = 500.0
    df_lep.loc[idx_dec, 'mvt_type_mouvement'] = 'APPORT'

    # 3. On crée la ligne d'intérêts pour Janvier
    # On repart de la ligne de décembre pour copier les IDs et paramètres
    row_interet = df_lep.loc[idx_dec].copy()
    row_interet['mvt_nb_parts'] = 412.0
    row_interet['mvt_type_mouvement'] = 'INTERET'
    
    # 4. On ajoute cette nouvelle ligne au tableau
    df_lep = pd.concat([df_lep, row_interet], ignore_index=True)

# 5. On n'oublie pas de retrier par date pour que SQL s'y retrouve
df_lep = df_lep.sort_values('mvt_date').reset_index(drop=True)

# On force la colonne en datetime (si ce n'est pas déjà fait)
df_lep['mvt_date'] = pd.to_datetime(df_lep['mvt_date'])

# On décale au dernier jour du mois
df_lep['mvt_date'] = df_lep['mvt_date'] + MonthEnd(0)

# On cible la ligne par sa date et son ID produit
mask = (df_lep['mvt_date'] == '2025-12-31') & (df_lep['pdt_id'] == 11)
# On change le type uniquement pour cette correspondance
df_lep.loc[mask, 'mvt_type_mouvement'] = 'INTERET'

In [103]:
df_lep

,mvt_date,mvt_prix,mvt_nb_parts,mvt_frais,mvt_type_mouvement,pdt_id
0,2024-01-31,1,12500.0,0,APPORT,11
1,2024-02-29,1,7500.0,0,APPORT,11
2,2024-09-30,1,-8500.0,0,RETRAIT,11
3,2024-12-31,1,500.0,0,APPORT,11
4,2024-12-31,1,412.0,0,INTERET,11
5,2025-02-28,1,1000.0,0,APPORT,11
6,2025-03-31,1,2000.0,0,APPORT,11
7,2025-06-30,1,500.0,0,APPORT,11
8,2025-07-31,1,500.0,0,APPORT,11
9,2025-10-31,1,3500.0,0,APPORT,11


In [104]:
df_lep.to_sql('mouvement_mvt', engine, if_exists='append', index=False)
print("LEP importé !")

LEP importé !
